# Evaluation Metrics Pipeline

This notebook runs an end-to-end evaluation workflow for two RAG strategies:

- **Strategy A**: Hybrid Retrieval (BM25 + Vector)
- **Strategy B**: Hybrid Retrieval + Rerank

It generates predictions, computes metrics against ground truth, and summarizes results.

## Metrics
- Retrieval: Recall@K, Precision@K, MRR@K, nDCG@K, MAP@K
- Generation: Exact Match, Token F1, ROUGE-L
- Citation (if available): Citation hit rate

In [1]:
from pathlib import Path
import subprocess
import json

try:
    import pandas as pd
except Exception:
    pd = None

def find_repo_root(start: Path) -> Path:
    p = start.resolve()
    for cur in [p] + list(p.parents):
        if (cur / "config.py").exists() and (cur / "scripts").exists() and (cur / "data").exists():
            return cur
    raise RuntimeError("Cannot find repository root")

ROOT = find_repo_root(Path.cwd())
print("Repository root:", ROOT)

Repository root: /Users/ben/Library/CloudStorage/GoogleDrive-rs.runsheng@gmail.com/My Drive/Colab Notebooks/CS614 GenAI/qa_agent_legal_tax


## 1) Install dependencies

Run once if needed.

In [5]:
def run(cmd):
    print("\n$", " ".join(cmd))
    subprocess.run(cmd, cwd=ROOT, check=True)

run(["python", "-m", "pip", "install", "-r", "requirements.txt"])


$ python -m pip install -r requirements.txt
  Using cached pydantic-2.5.0-py3-none-any.whl.metadata (174 kB)
  Using cached PyYAML-6.0.1-cp312-cp312-macosx_11_0_arm64.whl.metadata (2.1 kB)
  Using cached openai-1.3.0-py3-none-any.whl.metadata (16 kB)
  Using cached pandas-2.0.3.tar.gz (5.3 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [20 lines of output]
      Traceback (most recent call last):
        File "/opt/anaconda3/lib/python3.12/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 389, in <module>
          main()
        File "/opt/anaconda3/lib/python3.12/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        File "/opt/anaconda3/lib/python3.12/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 143, in get_requires_for_build_wheel
          return hook(config_settings)
                 ^^^^^^^^^^^^^^^^^^^^^
        File "/private/var/folders/ct/l9f0kpl97l79p90mfzn8ltwc0000gn/T/pip-build-env-9ed4o7vo/overlay/lib/python3.12/site-packages/setuptools/build_meta.py", 

CalledProcessError: Command '['python', '-m', 'pip', 'install', '-r', 'requirements.txt']' returned non-zero exit status 1.

## 2) Configure input/output paths

In [3]:
GT = ROOT / "data" / "qa_pairs" / "eval_ground_truth.jsonl"
if not GT.exists():
    raise FileNotFoundError(f"Ground truth file not found: {GT}")

PRED_HYBRID = ROOT / "outputs" / "preds_hybrid.jsonl"
PRED_HYBRID_RERANK = ROOT / "outputs" / "preds_hybrid_rerank.jsonl"

EVAL_HYBRID = ROOT / "outputs" / "eval_hybrid.json"
EVAL_HYBRID_RERANK = ROOT / "outputs" / "eval_hybrid_rerank.json"

EVAL_GEN_HYBRID = ROOT / "outputs" / "eval_gen_hybrid.json"
EVAL_GEN_HYBRID_RERANK = ROOT / "outputs" / "eval_gen_hybrid_rerank.json"

print("Ground truth:", GT)

FileNotFoundError: Ground truth file not found: /Users/ben/Library/CloudStorage/GoogleDrive-rs.runsheng@gmail.com/My Drive/Colab Notebooks/CS614 GenAI/qa_agent_legal_tax/data/qa_pairs/eval_ground_truth.jsonl

## 3) Generate predictions for both RAG strategies

In [ ]:
# Strategy A: Hybrid Retrieval (BM25 + Vector)
run([
    "python", "scripts/eval/run_eval_benchmark.py",
    "--gt", str(GT),
    "--out", str(PRED_HYBRID),
    "--enable-rerank", "false"
])

# Strategy B: Hybrid Retrieval + Rerank
run([
    "python", "scripts/eval/run_eval_benchmark.py",
    "--gt", str(GT),
    "--out", str(PRED_HYBRID_RERANK),
    "--enable-rerank", "true"
])

## 4) Compute evaluation metrics (retrieval + generation)

In [ ]:
run([
    "python", "scripts/eval/evaluate_predictions.py",
    "--gt", str(GT),
    "--pred", str(PRED_HYBRID),
    "--k", "5",
    "--out", str(EVAL_HYBRID)
])

run([
    "python", "scripts/eval/evaluate_predictions.py",
    "--gt", str(GT),
    "--pred", str(PRED_HYBRID_RERANK),
    "--k", "5",
    "--out", str(EVAL_HYBRID_RERANK)
])

## 5) Optional: generation-only metrics

In [ ]:
run([
    "python", "scripts/eval/evaluate_generation.py",
    "--gt", str(GT),
    "--pred", str(PRED_HYBRID),
    "--out", str(EVAL_GEN_HYBRID)
])

run([
    "python", "scripts/eval/evaluate_generation.py",
    "--gt", str(GT),
    "--pred", str(PRED_HYBRID_RERANK),
    "--out", str(EVAL_GEN_HYBRID_RERANK)
])

## 6) Load and compare results

In [ ]:
def load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))

hybrid = load_json(EVAL_HYBRID)
hybrid_rerank = load_json(EVAL_HYBRID_RERANK)

print("Hybrid:")
print(json.dumps(hybrid, indent=2, ensure_ascii=False))
print("\nHybrid + Rerank:")
print(json.dumps(hybrid_rerank, indent=2, ensure_ascii=False))

In [ ]:
def flatten_eval(label, obj):
    row = {"strategy": label}
    row.update(obj.get("retrieval", {}))
    gen = obj.get("generation", {})
    row.update({f"gen_{k}": v for k, v in gen.items()})
    return row

rows = [
    flatten_eval("Hybrid", hybrid),
    flatten_eval("Hybrid+Rerank", hybrid_rerank),
]

if pd is not None:
    display(pd.DataFrame(rows))
else:
    print(json.dumps(rows, indent=2, ensure_ascii=False))

## 7) Reproducibility artifacts

Keep these files in your report submission:

- `outputs/preds_hybrid.jsonl`
- `outputs/preds_hybrid_rerank.jsonl`
- `outputs/eval_hybrid.json`
- `outputs/eval_hybrid_rerank.json`
- `outputs/run_snapshot_YYYYmmdd_HHMMSS.json` (auto-generated by benchmark script)